In [3]:
# --- Install required packages ---
!pip install -q openpyxl

In [4]:
import os
import pandas as pd
import numpy as np
from collections import Counter

print("Files in session:", os.listdir())

Files in session: ['.config', 'tinyllama_results.xlsx', 'phi3_results.xlsx', 'qwen_results.xlsx', 'sample_data']


In [5]:
# --- Load model result files ---
df_qwen = pd.read_excel("qwen_results.xlsx")
df_tiny = pd.read_excel("tinyllama_results.xlsx")
df_phi3 = pd.read_excel("phi3_results.xlsx")

print("✅ Files loaded")
print(f"  Qwen      — {len(df_qwen)} rows | columns: {df_qwen.columns.tolist()}")
print(f"  TinyLlama — {len(df_tiny)} rows | columns: {df_tiny.columns.tolist()}")
print(f"  Phi-3     — {len(df_phi3)} rows | columns: {df_phi3.columns.tolist()}")

✅ Files loaded
  Qwen      — 10 rows | columns: ['Question_ID', 'Question', 'Qwen_Response', 'Qwen_Latency_s']
  TinyLlama — 10 rows | columns: ['Question_ID', 'Question', 'TinyLlama_Response', 'TinyLlama_Latency_s']
  Phi-3     — 10 rows | columns: ['Question_ID', 'Question', 'Phi3_Response', 'Phi3_Latency_s']


In [6]:
# --- All 3 files have real latency columns ---
print("✅ Qwen_Latency_s      — real values")
print("✅ TinyLlama_Latency_s — real values")
print("✅ Phi3_Latency_s      — real values")

✅ Qwen_Latency_s      — real values
✅ TinyLlama_Latency_s — real values
✅ Phi3_Latency_s      — real values


In [7]:
# --- Scoring functions ---

BANKING_KEYWORDS = [
    "kyc", "verification", "identity", "credit", "score",
    "loan", "emi", "interest", "insurance", "premium",
    "deposit", "account", "bank", "payment"
]

PROFESSIONAL_WORDS = ["customer", "bank", "policy", "financial", "payment"]

HALLUCINATION_MARKERS = [
    "as an ai", "i am an ai",
    "i'm sorry", "i am sorry",
    "i cannot", "i can't",
    "i don't know", "i do not know",
    "i don't have the ability", "i do not have",
    "user:", "customer:", "[user]", "[agent]", "<user>", "<agent>"
]


def evaluate_response(text):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0, 0, 0, 0, 0, 0

    text       = str(text)
    text_lower = text.lower()
    word_count = len(text.split())
    sentences  = [s.strip() for s in text.split(".") if s.strip()]

    # 1️⃣ Accuracy (1–5): BFSI keyword coverage
    keyword_hits = sum(1 for kw in BANKING_KEYWORDS if kw in text_lower)
    accuracy = min(max(keyword_hits, 1), 5)

    # 2️⃣ Relevance (1–5): stays on BFSI topic
    relevance = 5 if keyword_hits >= 2 else (3 if keyword_hits == 1 else 1)

    # 3️⃣ Clarity (1–5): proper sentence structure
    clarity = 5 if "." in text and word_count > 8 else (3 if word_count > 5 else 1)

    # 4️⃣ BFSI Tone (1–5): professional calling-agent language
    tone_hits = sum(1 for w in PROFESSIONAL_WORDS if w in text_lower)
    tone = min(tone_hits + 1, 5)

    # 5️⃣ Conciseness (1–5): appropriate length for a phone call
    if 12 <= word_count <= 80:
        conciseness = 5
    elif 6 <= word_count <= 120:
        conciseness = 3
    else:
        conciseness = 1

    # 6️⃣ Output Quality (1–5): penalise repetition, hallucinations, run-ons
    quality = 5
    if word_count < 5:
        quality -= 2
    word_freq = Counter(text_lower.split())
    repeated  = sum(1 for w, c in word_freq.items() if c > 3 and len(w) > 3)
    if repeated >= 2:
        quality -= 2
    elif repeated == 1:
        quality -= 1
    if any(m in text_lower for m in HALLUCINATION_MARKERS):
        quality -= 2
    if len(sentences) == 1 and word_count > 40:
        quality -= 1
    quality = max(quality, 1)

    return accuracy, relevance, clarity, tone, conciseness, quality


def latency_score(seconds):
    """Convert latency in seconds to 1–5 score. Lower is better."""
    if pd.isna(seconds): return 1
    if seconds <= 2:     return 5
    elif seconds <= 4:   return 4
    elif seconds <= 6:   return 3
    elif seconds <= 10:  return 2
    else:                return 1


def evaluate_model(df, response_col, latency_col, prefix):
    """Score a model dataframe across all 7 dimensions and return total."""
    SCORE_COLS = ["Accuracy", "Relevance", "Clarity", "Tone", "Conciseness", "Output_Quality"]

    scores = df[response_col].apply(evaluate_response)
    df[[f"{prefix}_{c}" for c in SCORE_COLS]] = pd.DataFrame(scores.tolist(), index=df.index)
    df[f"{prefix}_Latency_Score"] = df[latency_col].apply(latency_score)

    all_cols = [f"{prefix}_{c}" for c in SCORE_COLS] + [f"{prefix}_Latency_Score"]
    df[f"{prefix}_Total"] = df[all_cols].sum(axis=1)

    return df, SCORE_COLS


print("✅ Scoring functions defined (7 metrics: 6 quality + latency)")

✅ Scoring functions defined (7 metrics: 6 quality + latency)


In [8]:
# --- Evaluate all 3 models ---
df_qwen, SCORE_COLS = evaluate_model(df_qwen, "Qwen_Response",      "Qwen_Latency_s",      "Qwen")
df_tiny, SCORE_COLS = evaluate_model(df_tiny, "TinyLlama_Response", "TinyLlama_Latency_s", "Tiny")
df_phi3, SCORE_COLS = evaluate_model(df_phi3, "Phi3_Response",      "Phi3_Latency_s",      "Phi3")

print("--- Qwen Per-Question Scores ---")
print(df_qwen[["Question_ID", "Qwen_Total"]].to_string(index=False))
print(f"\nQwen Overall Total: {df_qwen['Qwen_Total'].sum()}")

print("\n--- TinyLlama Per-Question Scores ---")
print(df_tiny[["Question_ID", "Tiny_Total"]].to_string(index=False))
print(f"\nTinyLlama Overall Total: {df_tiny['Tiny_Total'].sum()}")

print("\n--- Phi-3 Per-Question Scores ---")
print(df_phi3[["Question_ID", "Phi3_Total"]].to_string(index=False))
print(f"\nPhi-3 Overall Total: {df_phi3['Phi3_Total'].sum()}")

--- Qwen Per-Question Scores ---
Question_ID  Qwen_Total
         Q1          28
         Q2          26
         Q3          27
         Q4          28
         Q5          22
         Q6          27
         Q7          25
         Q8          25
         Q9          25
        Q10          23

Qwen Overall Total: 256

--- TinyLlama Per-Question Scores ---
Question_ID  Tiny_Total
         Q1          26
         Q2          26
         Q3          28
         Q4          26
         Q5          22
         Q6          28
         Q7          26
         Q8          25
         Q9          23
        Q10          21

TinyLlama Overall Total: 251

--- Phi-3 Per-Question Scores ---
Question_ID  Phi3_Total
         Q1          27
         Q2          25
         Q3          27
         Q4          25
         Q5          22
         Q6          26
         Q7          25
         Q8          26
         Q9          24
        Q10          21

Phi-3 Overall Total: 248


In [9]:
# --- Summary ---
total_qwen = df_qwen["Qwen_Total"].sum()
total_tiny = df_tiny["Tiny_Total"].sum()
total_phi3 = df_phi3["Phi3_Total"].sum()

print("=" * 55)
print("           MODEL EVALUATION SUMMARY")
print("=" * 55)
print(f"{'Metric':<25} {'Qwen':>8} {'TinyLlama':>10} {'Phi-3':>8}")
print("-" * 55)
for c in SCORE_COLS:
    q_avg = df_qwen[f"Qwen_{c}"].mean()
    t_avg = df_tiny[f"Tiny_{c}"].mean()
    p_avg = df_phi3[f"Phi3_{c}"].mean()
    print(f"  {c:<23} {q_avg:>8.2f} {t_avg:>10.2f} {p_avg:>8.2f}")
print(f"  {'Latency':<23} {df_qwen['Qwen_Latency_Score'].mean():>8.2f} {df_tiny['Tiny_Latency_Score'].mean():>10.2f} {df_phi3['Phi3_Latency_Score'].mean():>8.2f}")
print("-" * 55)
print(f"  {'TOTAL SCORE':<23} {total_qwen:>8} {total_tiny:>10} {total_phi3:>8}")
print("=" * 55)

scores = {"Qwen": total_qwen, "TinyLlama": total_tiny, "Phi-3": total_phi3}
winner = max(scores, key=scores.get)
if list(scores.values()).count(scores[winner]) > 1:
    print("🤝 Overall Result: Tie")
else:
    print(f"🏆 Overall Winner: {winner}")

           MODEL EVALUATION SUMMARY
Metric                        Qwen  TinyLlama    Phi-3
-------------------------------------------------------
  Accuracy                    3.00       2.70     2.50
  Relevance                   4.60       4.60     4.60
  Clarity                     5.00       5.00     5.00
  Tone                        1.90       1.70     1.70
  Conciseness                 5.00       5.00     5.00
  Output_Quality              4.90       4.20     5.00
  Latency                     1.20       1.90     1.00
-------------------------------------------------------
  TOTAL SCORE                  256        251      248
🏆 Overall Winner: Qwen


In [11]:
# --- Save each model's evaluated results independently ---
df_qwen.to_excel("qwen_evaluated.xlsx",      index=False)
df_tiny.to_excel("tinyllama_evaluated.xlsx", index=False)
df_phi3.to_excel("phi3_evaluated.xlsx",      index=False)

print("✅ Qwen evaluation saved      → qwen_evaluated.xlsx")
print("✅ TinyLlama evaluation saved → tinyllama_evaluated.xlsx")
print("✅ Phi-3 evaluation saved     → phi3_evaluated.xlsx")

✅ Qwen evaluation saved      → qwen_evaluated.xlsx
✅ TinyLlama evaluation saved → tinyllama_evaluated.xlsx
✅ Phi-3 evaluation saved     → phi3_evaluated.xlsx
